# AI Meeting Minutes Professional Notebook


#### Imports

In [1]:
import torch              # For hardware acceleration (GPU/CPU)
import gradio as gr       # For creating the web user interface
from faster_whisper import WhisperModel  # For high-speed audio transcription
from openai import OpenAI # To connect with Ollama (OpenAI compatible API)
from jiwer import wer, cer # For calculating Word Error Rate and Character Error Rate

#### Configuration and Settings

In [2]:
# --- Model Settings ---
WHISPER_SIZE = "medium"              # Size of the Whisper model (base, small, medium, large-v3)
AI_MODEL_NAME = "gpt-oss:120b-cloud" # The Ollama model to use for summarization
TEXT_CHUNK_SIZE = 12000              # Max characters to process in one AI request
MAX_PROMPT_SIZE = 30000               # Limit for combined summaries before condensing

# --- Hardware Settings ---
IS_GPU_AVAILABLE = torch.cuda.is_available() # Check if NVIDIA GPU is available
# Use float16 for GPU to save memory/speed, else use float32 for CPU
COMPUTE_TYPE = "int8_float16" if IS_GPU_AVAILABLE else "float32"
DEVICE = "cuda" if IS_GPU_AVAILABLE else "cpu"

# --- AI Client Setup ---
# Connecting to local Ollama instance
ai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

# --- Prompts ---
# This prompt is used for analyzing small sections of the transcript
CHUNK_SYSTEM_PROMPT = """You are a professional scribe. Extract ALL critical information.
Capture every decision, action item, and key argument. Maintain timestamps.
Output detailed notes for this section."""

# This prompt is used for the final professional meeting minutes
FINAL_SYSTEM_PROMPT = """You are an expert corporate secretary. Consolidate these summaries into professional meeting minutes.

STRICT RULES:
1. Under '# Meeting Duration', write the exact duration provided.
2. Output ONLY Markdown.
3. Combine similar topics into cohesive discussion points.
4. Ensure NO action item or decision is lost.

Use EXACT headings:
# Meeting Summary
# Meeting Duration
# Attendees
# Agenda
# Key Discussion Points
# Decisions Made
# Action Items
# Votes / Motions
# Risks / Blockers
# Next Steps
""".strip()

#### Helper Functions

In [3]:
def load_whisper_model():
    """Initializes and loads the faster-whisper model into memory."""
    return WhisperModel(WHISPER_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)

# Load the model once when the notebook starts
whisper_model = load_whisper_model()

def format_time(seconds_value):
    """Converts raw seconds (e.g., 125) into a readable format (e.g., 02:05)."""
    seconds_value = int(seconds_value)
    minutes, seconds = divmod(seconds_value, 60)
    
    if minutes >= 60:
        hours, minutes = divmod(minutes, 60)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    
    return f"{minutes:02d}:{seconds:02d}"

def get_audio_text(audio_file):
    """Transcribes the audio file and returns timestamped text, plain text, and duration."""
    # beam_size=5 provides a good balance between speed and accuracy
    segments, info = whisper_model.transcribe(audio_file, beam_size=5)
    
    text_with_time = [] 
    plain_text_list = [] 
    
    for segment in segments:
        timestamp = format_time(segment.start)
        text = segment.text.strip()
        
        if text:
            text_with_time.append(f"[{timestamp}] {text}")
            plain_text_list.append(text)
            
    total_duration = format_time(info.duration)
    return "\n".join(text_with_time), " ".join(plain_text_list), total_duration

def ask_ai_model(system_prompt, user_content):
    """Sends a prompt to the Ollama AI model and returns the text response."""
    response = ai_client.chat.completions.create(
        model=AI_MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        temperature=0 # 0 temperature ensures consistent, factual responses
    )
    return response.choices[0].message.content

def check_text_quality(reference_text, ai_text):
    """Calculates Word Error Rate (WER) and Character Error Rate (CER) to check accuracy."""
    if not reference_text or not ai_text:
        return "No reference text provided. Quality metrics skipped."
    
    # Convert to lowercase for fair comparison
    ref_clean = reference_text.lower().strip()
    ai_clean = ai_text.lower().strip()
    
    word_error_rate = wer(ref_clean, ai_clean) * 100
    char_error_rate = cer(ref_clean, ai_clean) * 100
    
    # Assign a status based on the WER percentage
    if word_error_rate < 10:
        status = "⭐ Excellent"
    elif word_error_rate < 20:
        status = "👍 Good"
    else:
        status = "⚠️ Review required"
        
    return f"WER: {word_error_rate:.2f}%\nCER: {char_error_rate:.2f}%\n\nStatus: {status}"

#### Main Pipeline Logic

In [4]:
def run_meeting_pipeline(audio_file, reference_text, progress=gr.Progress()):
    """Handles the end-to-end process: Audio -> Transcript -> Quality Check -> AI Summary -> Minutes."""
    
    # 1. Validate input
    if audio_file is None:
        yield "❌ Please upload or record audio.", "", "", ""

    # 2. Transcription Phase
    yield "🚀 Transcribing...", "", "", ""
    progress(0.2, desc="Transcribing Audio...")
    try:
        full_transcript, clean_transcript, total_duration = get_audio_text(audio_file)
    except Exception as e:
        yield f"❌ Transcription Error: {str(e)}", "", "", ""
        return

    # 3. Quality Evaluation Phase
    yield "📊 Calculating quality metrics...", full_transcript, "", ""
    progress(0.4, desc="Evaluating...")
    metrics_result = check_text_quality(reference_text, clean_transcript)

    # 4. Chunk-based Analysis Phase (to handle very long meetings)
    yield "✍️ Analyzing transcript in chunks...", full_transcript, metrics_result, ""
    
    # Split the transcript into smaller parts
    transcript_chunks = [
        full_transcript[i : i + TEXT_CHUNK_SIZE] 
        for i in range(0, len(full_transcript), TEXT_CHUNK_SIZE)
    ]
    
    chunk_summaries = []
    for idx, chunk in enumerate(transcript_chunks):
        # Update progress bar for each chunk
        progress_val = 0.4 + (0.4 * (idx + 1) / len(transcript_chunks))
        progress(progress_val, desc=f"Processing chunk {idx+1}/{len(transcript_chunks)}...")
        
        # Get summary for this specific chunk
        summary = ask_ai_model(CHUNK_SYSTEM_PROMPT, f"Chunk {idx+1} of {len(transcript_chunks)}:\n{chunk}")
        chunk_summaries.append(summary)

    combined_summaries = "\n\n--- Section ---\n\n".join(chunk_summaries)
    
    # 5. Condensing Phase (if the combined summary is still too long for the AI)
    if len(combined_summaries) > MAX_PROMPT_SIZE:
        yield "🌀 Condensing summaries for long meeting...", full_transcript, metrics_result, ""
        summary_chunks = [
            combined_summaries[i : i + MAX_PROMPT_SIZE] 
            for i in range(0, len(combined_summaries), MAX_PROMPT_SIZE)
        ]
        condensed_list = []
        for s_chunk in summary_chunks:
            condensed_list.append(ask_ai_model("Summarize this section while keeping key decisions.", s_chunk))
        combined_summaries = "\n\n".join(condensed_list)

    # 6. Final Synthesis Phase
    yield "🪄 Finalizing global meeting minutes...", full_transcript, metrics_result, ""
    
    # Provide the AI with the total duration and the combined summaries
    final_user_input = f"--- MANDATORY DATA ---\nTOTAL MEETING DURATION: {total_duration}\n--- END DATA ---\n\nSUMMARIES:\n{combined_summaries}"
    
    try:
        final_response = ai_client.chat.completions.create(
            model=AI_MODEL_NAME,
            messages=[
                {"role": "system", "content": FINAL_SYSTEM_PROMPT},
                {"role": "user", "content": final_user_input},
            ],
            temperature=0,
            stream=False 
        )
        final_minutes = final_response.choices[0].message.content
    except Exception as e:
        yield f"❌ Final Synthesis Error: {str(e)}", full_transcript, metrics_result, ""
        return

    progress(1.0, desc="Done!")
    yield "✅ All Process Completed!", full_transcript, metrics_result, final_minutes

#### Gradio UI Interface

In [5]:
# Use Gradio Blocks for a custom layout
with gr.Blocks(title="AI Meeting Minutes Pro") as demo:
    gr.Markdown("# 📝 AI Meeting Minutes Professional")
    gr.Markdown("Optimized for very long meetings.")

    with gr.Tabs():
        # Tab 1: Input and Trigger
        with gr.Tab("1. Setup & Generate"):
            with gr.Row():
                with gr.Column():
                    audio_input = gr.Audio(
                        label="Meeting Audio File", 
                        type="filepath", 
                        sources=["upload", "microphone"]
                    )
                    ref_input = gr.Textbox(label="Reference Transcript (Optional)", lines=5)
                    generate_btn = gr.Button("Start Processing", variant="primary")
                    status_box = gr.Textbox(label="Current Status", value="Ready", interactive=False)

        # Tab 2: Raw Output
        with gr.Tab("2. Full Transcript"):
            transcript_display = gr.Textbox(label="AI Generated Transcript", lines=20, interactive=False)
        
        # Tab 3: Quality Check
        with gr.Tab("3. Quality Metrics"):
            metrics_display = gr.Textbox(label="WER/CER Analysis", lines=5, interactive=False)
        
        # Tab 4: Final Result
        with gr.Tab("4. Meeting Minutes"):
            minutes_display = gr.Markdown(label="Final Minutes")

    # Link the button to the pipeline function
    generate_btn.click(
        fn=run_meeting_pipeline,
        inputs=[audio_input, ref_input],
        outputs=[status_box, transcript_display, metrics_display, minutes_display]
    )

# Launch the app
if __name__ == "__main__":
    demo.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
